# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library. All dataset entities are referenced by their `@id` for full reproducibility.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs used in this dataset.

Each entity (record set, field, column) is referenced below by its `@id`.

In [ ]:
# List all record sets and their fields
print('Available record sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Number of fields: {len(rs.fields)}")
    for field in rs.fields:
        print(f"    - Field: {field.name} (@id: {field.id}) type: {field.data_type}")
    print('---')

## 3. Data Extraction
Load data from each record set into separate pandas DataFrames for further analysis.

Below, we extract data using the record set and field `@id`s observed above.

In [ ]:
# Build a mapping of record set @id to name for easy reference, and extract dataframes
dfs = {}

for rs in dataset.record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dfs[rs.id] = df
    print(f"Loaded RecordSet '{rs.name}' (@id: {rs.id}) with {len(df)} records.")

# Print all record set @ids and first few columns
print("\nRecordSet @id list:")
for rs_id, df in dfs.items():
    print(f"@id: {rs_id}, Columns: {list(df.columns)[:5]} ... ({len(df.columns)} total columns)")

# For demonstration, pick the first record set for further processing
main_record_set_id = list(dfs.keys())[0]
print(f"\nSample data from main RecordSet (@id: {main_record_set_id}):")
display(dfs[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's explore the dataset's main record set by filtering, normalizing, and grouping using field `@id`s only.

We'll select a numeric field and a group/categorical field based on the columns present.

In [ ]:
df = dfs[main_record_set_id]

# Find numeric fields by @id (i.e., column names)
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field found in this record set.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by a categorical field (@id), e.g., first string or boolean column
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]):
            group_field_id = col
            break
    if group_field_id:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped.head())

## 5. Visualization
Let's visualize the distribution of the main numeric field in this record set and, if available, the grouped values by category.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color='#3274a1')
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion

In this notebook, we loaded and explored the clinicopathological and molecular dataset of second primary colorectal cancer survivors using `mlcroissant`. We reviewed its Croissant-defined record sets and fields using their `@id`, loaded the main tabular record set, performed numeric filtering and normalization, grouped the data by a categorical field, and visualized the field distributions. The dataset structure makes rigorous and reproducible analysis possible, as every column and field can be referenced by its unique `@id`.

**Next steps:** You can extend this analysis with more detailed modeling, statistical tests, or visualization using the referenced `@id`s for all data elements, fully leveraging the FAIR data principles.